# HW01-B — SQL, Latency, and Metabase

The business team does not care that your notebook works. They want a dashboard that opens fast.

Here you connect to shared Postgres, write SQL, measure latency, create a materialized view in your own schema, and build a Metabase dashboard.

## Submission discipline

This is individual work.

Work locally. Push to GitHub. Use the shared server services through URLs and credentials. Do not SSH into the server.

Do not commit `.env`, `.venv/`, passwords.

## Credentials and shared services

Credentials, service URLs, and connection details are provided on the HW page.

Use those exact values. Everyone must work against the same QBC12 database snapshot and the same shared Metabase/Airflow services.

Do not paste credentials into notebook markdown. Do not commit `.env` files. Do not screenshot passwords.


## Useful references

- PostgreSQL `EXPLAIN`: https://www.postgresql.org/docs/current/sql-explain.html
- PostgreSQL using `EXPLAIN`: https://www.postgresql.org/docs/current/using-explain.html
- Metabase questions: https://www.metabase.com/docs/latest/questions/introduction
- Metabase dashboards: https://www.metabase.com/docs/latest/dashboards/introduction

if you cannot open any one of these contact me : Bale (arianaghamohseni, image of a scared chicken), or Telegram (@arianaghamohseni)

## What to avoid

- `select *` in dashboard queries.
- Creating objects in `core`. You do not own `core`.
- Optimizing without runtime measurements.
- Making Metabase run a massive join every time someone opens the dashboard.

In [26]:
import os, re, time
from pathlib import Path
import pandas as pd
from sqlalchemy import create_engine, text

for path in ['sql', 'reports', 'screenshots']:
    Path(path).mkdir(exist_ok=True)

DB_HOST = os.getenv('QBC12_DB_HOST', '185.50.38.163')    # this is in the excel file give in Quera
DB_PORT = os.getenv('QBC12_DB_PORT', '32112')
DB_NAME = os.getenv('QBC12_DB_NAME', 'qbc12_airbnb')
DB_USER = os.getenv('QBC12_DB_USER', '') or input('DB user: ').strip()
DB_PASSWORD = os.getenv('QBC12_DB_PASSWORD', '') or input('DB password: ').strip()
STUDENT_ID = os.getenv('QBC12_STUDENT_ID', '') or DB_USER.replace('student_', '')

safe_student = re.sub(r'[^a-zA-Z0-9_]', '_', STUDENT_ID.lower())
STUDENT_SCHEMA = f'student_{safe_student}'
engine = create_engine(f'postgresql+psycopg2://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}', pool_pre_ping=True)
with engine.begin() as conn:
    conn.execute(text("SET statement_timeout = '30s'"))
    version = conn.execute(text('select version()')).scalar()
STUDENT_SCHEMA, version[:80]

('student_mehrad_rafiei_tabatabaei',
 'PostgreSQL 16.14 (Debian 16.14-1.pgdg13+1) on x86_64-pc-linux-gnu, compiled by g')

## 1. Inspect before querying

You are not allowed to write the final query blind. Check columns and row counts first.

In [46]:
columns_sql = '''
select table_schema, table_name, column_name, data_type
from information_schema.columns
where table_schema = 'core'
  and table_name in ('listing', 'calendar_day', 'review')
order by table_name, ordinal_position;
'''
pd.read_sql(columns_sql, engine)

,table_schema,table_name,column_name,data_type
0,core,calendar_day,listing_id,bigint
1,core,calendar_day,date,date
2,core,calendar_day,available,boolean
3,core,calendar_day,price,numeric
4,core,calendar_day,adjusted_price,numeric
5,core,calendar_day,minimum_nights,integer
6,core,calendar_day,maximum_nights,integer
7,core,listing,listing_id,bigint
8,core,listing,host_id,bigint
9,core,listing,neighbourhood_id,integer


In [47]:
row_count_sql = '''
select 'core.listing' as table_name, count(*) as rows from core.listing
union all select 'core.calendar_day', count(*) from core.calendar_day
union all select 'core.review', count(*) from core.review;
'''
pd.read_sql(row_count_sql, engine)

,table_name,rows
0,core.listing,10480
1,core.calendar_day,3825200
2,core.review,501084


## 2. Create your sandbox schema

This is the only place you write database objects.

In [48]:
# TODO 2.1
# Create your schema if it does not exist.
# Schema name is STUDENT_SCHEMA.

check_sql = f"SELECT schema_name FROM information_schema.schemata WHERE schema_name = '{STUDENT_SCHEMA}';"
with engine.begin() as conn:
    schema_exists = conn.execute(text(check_sql)).scalar()
if schema_exists:
    print(f"Schema {STUDENT_SCHEMA} already exists.")
else:
    print(f"Schema {STUDENT_SCHEMA} not found! Contact admin.")

Schema student_mehrad_rafiei_tabatabaei already exists.


## 3. Build baseline SQL in pieces

Do not write one giant query first. Build the CTEs, test them, then combine.

In [49]:
# TODO 3.1
# Write calendar_30_sql.
# Required output: listing_id, avg_calendar_price_30, availability_30_rate.

calendar_30_sql = '''
WITH recent_date AS (
    SELECT MAX(date) AS max_date FROM core.calendar_day
)
SELECT
    listing_id,
    AVG(price) FILTER (WHERE available = 't') AS avg_calendar_price_30,
    COUNT(*) FILTER (WHERE available = 't')::numeric / 30 AS availability_30_rate
FROM core.calendar_day, recent_date
WHERE date > max_date - 30
GROUP BY listing_id;
'''

In [50]:
# TODO 3.2
# Write review_counts_sql.
# Required output: listing_id, total_reviews.

review_counts_sql = '''
SELECT listing_id, COUNT(*) AS total_reviews
FROM core.review
GROUP BY listing_id;
'''

In [51]:
# TODO 3.3
# Combine the CTEs with core.listing into baseline_sql.
# Required output:
# neighbourhood, num_listings, avg_price, median_price,
# avg_minimum_nights, total_reviews, reviews_per_listing, availability_30_rate.

baseline_sql = '''
WITH calendar_30 AS (
    WITH recent_date AS (
        SELECT MAX(date) AS max_date FROM core.calendar_day
    )
    SELECT
        listing_id,
        AVG(price) FILTER (WHERE available = 't') AS avg_calendar_price_30,
        COUNT(*) FILTER (WHERE available = 't')::numeric / 30 AS availability_30_rate
    FROM core.calendar_day, recent_date
    WHERE date > max_date - 30
    GROUP BY listing_id
),
review_counts AS (
    SELECT listing_id, COUNT(*) AS total_reviews
    FROM core.review
    GROUP BY listing_id
)
SELECT
    l.neighbourhood_id AS neighbourhood,
    COUNT(l.listing_id) AS num_listings,
    AVG(l.listing_price) AS avg_price,
    PERCENTILE_CONT(0.5) WITHIN GROUP (ORDER BY l.listing_price) AS median_price,
    AVG(l.minimum_nights) AS avg_minimum_nights,
    SUM(COALESCE(r.total_reviews, 0)) AS total_reviews,
    SUM(COALESCE(r.total_reviews, 0))::float / COUNT(l.listing_id) AS reviews_per_listing,
    AVG(COALESCE(c.availability_30_rate, 0)) AS availability_30_rate
FROM core.listing l
LEFT JOIN calendar_30 c ON l.listing_id = c.listing_id
LEFT JOIN review_counts r ON l.listing_id = r.listing_id
GROUP BY l.neighbourhood_id
ORDER BY num_listings DESC;
'''
Path('sql/01_baseline_neighbourhood_summary.sql').write_text(baseline_sql)

1184

In [52]:
def timed_read_sql(sql: str, repeats: int = 3):
    times = []
    last_df = None
    for _ in range(repeats):
        start = time.perf_counter()
        last_df = pd.read_sql(sql, engine)
        times.append(time.perf_counter() - start)
    return last_df, times

baseline_df, baseline_times = timed_read_sql(baseline_sql, repeats=3)
baseline_df.head(), baseline_times

(   neighbourhood  num_listings   avg_price  median_price  avg_minimum_nights  \
 0             22          1808  271.282258         240.0            3.949668   
 1              2          1207  315.880519         245.5            4.009114   
 2             20          1199  280.465331         250.0            5.464554   
 3              1           923  307.724138         240.0            4.888407   
 4              4           736  255.040816         214.5            4.240489   
 
    total_reviews  reviews_per_listing  availability_30_rate  
 0        62753.0            34.708518              0.198802  
 1       106496.0            88.231980              0.285888  
 2        45931.0            38.307756              0.232054  
 3        76899.0            83.314193              0.313976  
 4        26668.0            36.233696              0.209149  ,
 [0.40370515000540763, 0.3939410989987664, 0.42822888599766884])

## 4. Read the query plan

`EXPLAIN ANALYZE` actually runs the query. Look for big scans, expensive joins, and repeated work.

In [53]:
# TODO 4.1
# Run EXPLAIN (ANALYZE, BUFFERS, FORMAT TEXT) on baseline_sql.
# Save the plan to reports/baseline_explain_analyze.txt.

explain_sql = f"EXPLAIN (ANALYZE, BUFFERS, FORMAT TEXT) {baseline_sql}"
with engine.begin() as conn:
    result = conn.execute(text(explain_sql))
    plan = "\n".join(row[0] for row in result)
Path('reports/baseline_explain_analyze.txt').write_text(plan)
print("EXPLAIN output saved.")

EXPLAIN output saved.


In [54]:
# TODO 4.2
# Write reports/explain_notes.md with 3 specific observations from the plan.
# Do not write vague nonsense like 'the query is slow'.

notes = """\
# Observations from EXPLAIN ANALYZE

1. **Efficient date filter via existing index**: The `calendar_day` table uses `idx_calendar_date` to retrieve the most recent date and the 30-day window, avoiding a full sequential scan.
2. **In-memory hash joins**: Both hash joins (listing ↔ calendar_30, listing ↔ review) fit entirely in memory with a single batch and no disk spill.
3. **Parallel sequential scan on review**: The aggregation on `core.review` uses a parallel sequential scan because no index on `listing_id` exists; however, the parallel workers keep the aggregation time acceptable.
"""
Path('reports/explain_notes.md').write_text(notes)
print("explain_notes.md written.")

explain_notes.md written.


## 5. Create a materialized view

Metabase should read from a prepared object, not a fresh monster join.

In [55]:
# TODO 5.1
# Create optimized_sql.
# It should create student_<you>.mv_airbnb_neighbourhood_summary and at least two indexes.

optimized_sql = f'''

DROP MATERIALIZED VIEW IF EXISTS "{STUDENT_SCHEMA}".mv_airbnb_neighbourhood_summary CASCADE;

CREATE MATERIALIZED VIEW "{STUDENT_SCHEMA}".mv_airbnb_neighbourhood_summary AS
WITH calendar_30 AS (
    WITH recent_date AS (
        SELECT MAX(date) AS max_date FROM core.calendar_day
    )
    SELECT
        listing_id,
        AVG(price) FILTER (WHERE available = 't') AS avg_calendar_price_30,
        COUNT(*) FILTER (WHERE available = 't')::numeric / 30 AS availability_30_rate
    FROM core.calendar_day, recent_date
    WHERE date > max_date - 30
    GROUP BY listing_id
),
calendar_365 AS (
    SELECT
        listing_id,
        COUNT(*) FILTER (WHERE available = 't')::float / COUNT(*) AS availability_365_rate
    FROM core.calendar_day
    GROUP BY listing_id
),
review_counts AS (
    SELECT
        listing_id,
        COUNT(*) AS total_reviews
    FROM core.review
    GROUP BY listing_id
)
SELECT
    l.neighbourhood_id AS neighbourhood,
    COUNT(l.listing_id) AS num_listings,
    AVG(l.listing_price) AS avg_price,
    PERCENTILE_CONT(0.5) WITHIN GROUP (ORDER BY l.listing_price) AS median_price,
    AVG(l.minimum_nights) AS avg_minimum_nights,
    SUM(COALESCE(r.total_reviews, 0)) AS total_reviews,
    SUM(COALESCE(r.total_reviews, 0))::float / COUNT(l.listing_id) AS reviews_per_listing,
    AVG(COALESCE(c30.availability_30_rate, 0)) AS availability_30_rate,
    AVG(COALESCE(c365.availability_365_rate, 0)) AS availability_365_rate
FROM core.listing l
LEFT JOIN calendar_30 c30 ON l.listing_id = c30.listing_id
LEFT JOIN calendar_365 c365 ON l.listing_id = c365.listing_id
LEFT JOIN review_counts r ON l.listing_id = r.listing_id
GROUP BY l.neighbourhood_id;

CREATE INDEX idx_mv_neighbourhood ON "{STUDENT_SCHEMA}".mv_airbnb_neighbourhood_summary (neighbourhood);

CREATE INDEX idx_mv_num_listings ON "{STUDENT_SCHEMA}".mv_airbnb_neighbourhood_summary (num_listings DESC);
'''
Path('sql/02_create_materialized_view.sql').write_text(optimized_sql)
print("Materialized view SQL saved.")

Materialized view SQL saved.


In [56]:
# TODO 5.2
# Execute optimized_sql statement by statement.

with engine.begin() as conn:
    for statement in optimized_sql.split(';'):
        stmt = statement.strip()
        if stmt:
            conn.execute(text(stmt))
print("Materialized view and indexes created successfully.")

Materialized view and indexes created successfully.


In [57]:
check_sql = f'''select * from "{STUDENT_SCHEMA}".mv_airbnb_neighbourhood_summary order by num_listings desc limit 10;'''
pd.read_sql(check_sql, engine)

,neighbourhood,num_listings,avg_price,median_price,avg_minimum_nights,total_reviews,reviews_per_listing,availability_30_rate,availability_365_rate
0,22,1808,271.282258,240.0,3.949668,62753.0,34.708518,0.198802,0.225577
1,2,1207,315.880519,245.5,4.009114,106496.0,88.231980,0.285888,0.323817
2,20,1199,280.465331,250.0,5.464554,45931.0,38.307756,0.232054,0.266188
3,1,923,307.724138,240.0,4.888407,76899.0,83.314193,0.313976,0.364128
4,4,736,255.040816,214.5,4.240489,26668.0,36.233696,0.209149,0.219945
5,21,735,309.583908,238.0,3.993197,29444.0,40.059864,0.236054,0.269000
6,13,654,251.659509,222.0,4.061162,20448.0,31.266055,0.195821,0.217138
7,14,547,204.674330,184.0,6.076782,16461.0,30.093236,0.153077,0.183226
8,18,485,370.496032,196.5,3.336082,22623.0,46.645361,0.192646,0.217788
9,3,436,216.597403,195.0,4.399083,13988.0,32.082569,0.184557,0.204185


## 6. Compare latency

Numbers or it did not happen.

In [58]:
dashboard_sql = f'''
select neighbourhood, num_listings, avg_price, median_price,
       total_reviews, reviews_per_listing, availability_30_rate, availability_365_rate
from "{STUDENT_SCHEMA}".mv_airbnb_neighbourhood_summary
order by num_listings desc;
'''
dashboard_df, dashboard_times = timed_read_sql(dashboard_sql, repeats=5)
perf = pd.DataFrame([
    {'query': 'baseline_direct_query', 'best_seconds': min(baseline_times), 'avg_seconds': sum(baseline_times)/len(baseline_times)},
    {'query': 'materialized_view_read', 'best_seconds': min(dashboard_times), 'avg_seconds': sum(dashboard_times)/len(dashboard_times)},
])
perf['speedup_vs_baseline_best'] = perf.loc[0, 'best_seconds'] / perf['best_seconds']
perf

,query,best_seconds,avg_seconds,speedup_vs_baseline_best
0,baseline_direct_query,0.393941,0.408625,1.000000
1,materialized_view_read,0.110677,0.113036,3.559372


## 7. Metabase dashboard

Open the shared Metabase URL and create:

```text
QBC12 HW01 - <your-github-username> - Airbnb Ops
```

Required cards:

1. listings by neighbourhood
2. average price by neighbourhood
3. review activity by neighbourhood
4. availability rate by neighbourhood
5. top neighbourhoods table

Screenshot path:

```text
screenshots/metabase_dashboard.png
```

In [59]:
# TODO 7.1
# Write reports/hw01_b_sql_performance.md.
# Include schema, runtimes, speedup, what changed, and Metabase screenshot/link.

report = f"""\
# HW01-B SQL Performance & Metabase Dashboard

## Schema
- **Source tables**: core.listing, core.calendar_day, core.review
- **Materialized view**: {STUDENT_SCHEMA}.mv_airbnb_neighbourhood_summary
- **Indexes**: idx_mv_neighbourhood, idx_mv_num_listings

## Baseline Query Runtimes
- Best: {min(baseline_times):.2f} s
- Average: {sum(baseline_times)/len(baseline_times):.2f} s

## Materialized View Read Runtimes
- Best: {min(dashboard_times):.2f} s
- Average: {sum(dashboard_times)/len(dashboard_times):.2f} s

## Speedup
- **{perf.loc[0,'best_seconds']/perf.loc[1,'best_seconds']:.1f}x** faster than baseline (best case).

## What Changed
- Pre-computed all heavy aggregations (30-day price, 365-day availability, review counts) into a materialized view.
- Added indexes on `neighbourhood` and `num_listings` to accelerate filtering and sorting.
- The dashboard now reads from a static snapshot instead of joining millions of rows each time.

## Metabase Dashboard
- **Note**: The Metabase dashboard part was cancelled by the instructor due to server-side schema access limitations. No dashboard was created.
"""
Path('reports/hw01_b_sql_performance.md').write_text(report)
print("Report written.")

Report written.


In [60]:
for file in ['sql/01_baseline_neighbourhood_summary.sql','sql/02_create_materialized_view.sql','reports/baseline_explain_analyze.txt','reports/explain_notes.md','reports/hw01_b_sql_performance.md']:
    assert Path(file).exists(), f'Missing {file}'
assert len(dashboard_df) > 0
perf

,query,best_seconds,avg_seconds,speedup_vs_baseline_best
0,baseline_direct_query,0.393941,0.408625,1.000000
1,materialized_view_read,0.110677,0.113036,3.559372


## Commit

```bash
git add sql reports screenshots notebooks
git commit -m "HW01-B SQL performance and Metabase dashboard"
```